# Deloitte Quantum Sustainability Challenge 2026 — Final Submission

**Team:** Akhil Kambhatla, University of Maryland, College Park

## Pipeline Overview
1. Feature engineering from wildfire, weather, insurance, and census data (2018–2022)
2. Classical baselines: XGBoost and Random Forest (Task 1A)
3. Five quantum model variants including a novel domain-informed circuit (Task 1A)
4. 2023 wildfire risk predictions for all California zip codes (Task 1A)
5. Classical vs quantum evaluation and ablation study (Task 1B)
6. Insurance premium prediction with quantum fire risk features (Task 2)

In [1]:
# === Cell 1: Imports ===
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix, mean_absolute_error,
                             mean_squared_error, r2_score)
from xgboost import XGBClassifier, XGBRegressor
from imblearn.under_sampling import RandomUnderSampler

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_algorithms.optimizers import COBYLA

print(f"Platform: Qiskit {__import__('qiskit').__version__}")
print(f"Simulator: Qiskit Aer (statevector)")

Platform: Qiskit 2.3.1
Simulator: Qiskit Aer (statevector)


---
## Part 1: Feature Engineering (2018–2022)

### 2022 Data Handling
The wildfire dataset contains full weather coverage for 2018–2021 (~2,595 zip codes per year with both fire and non-fire examples). For 2022, only 173 fire-event zip codes are present with no weather rows or negative examples.

**Approach:** For 2022, we carry forward 2021 weather, insurance, and census features for all zip codes. Fire history features are computed properly using 2018–2021 fire records. The 173 known 2022 fire zips are labeled positive; all remaining zips are labeled negative. This assumption is valid because the dataset exhaustively lists fire events — absence from the list indicates no fire occurred.

In [2]:
# === Cell 2: Load Raw Data ===

wildfire = pd.read_csv('../data/wildfire_weather.csv')
insurance = pd.read_csv('../data/insurance_fire_census_weather_raw.csv')

wildfire['derived_year'] = wildfire['year_month'].str[:4].astype(float)

fire_events = wildfire[wildfire['FIRE_NAME'].notna()].copy()
fire_events['Year'] = fire_events['Year'].fillna(fire_events['derived_year'])

print(f"Wildfire dataset: {wildfire.shape[0]:,} rows")
print(f"Fire events: {len(fire_events):,}")
print(f"Insurance dataset: {insurance.shape[0]:,} rows")
print(f"\nFire events by year:")
print(fire_events.groupby('Year')['FIRE_NAME'].count())

Wildfire dataset: 125,476 rows
Fire events: 2,211
Insurance dataset: 47,033 rows

Fire events by year:
Year
2018.0    412
2019.0    319
2020.0    502
2021.0    388
2022.0    306
2023.0    284
Name: FIRE_NAME, dtype: int64


In [3]:
# === Cell 3: Build Base Table (2018–2022) ===

weather_rows = wildfire[wildfire['avg_tmax_c'].notna()].copy()
weather_rows['year'] = weather_rows['derived_year']

base_2018_2021 = weather_rows[weather_rows['year'].isin([2018, 2019, 2020, 2021])]\
    .groupby(['zip', 'year']).size().reset_index()[['zip', 'year']]

zips_2021 = base_2018_2021[base_2018_2021['year'] == 2021]['zip'].unique()
base_2022 = pd.DataFrame({'zip': zips_2021, 'year': 2022.0})
base_table = pd.concat([base_2018_2021, base_2022], ignore_index=True)
base_table = base_table.sort_values(['zip', 'year']).reset_index(drop=True)

print(f"Base table: {len(base_table):,} zip-year rows")
print(f"Per year: {base_table.groupby('year').size().to_dict()}")

Base table: 12,965 zip-year rows
Per year: {2018.0: 2593, 2019.0: 2593, 2020.0: 2593, 2021.0: 2593, 2022.0: 2593}


In [4]:
# === Cell 4: Target Variable ===

fire_zip_years = set()
for _, row in fire_events.iterrows():
    if pd.notna(row['zip']) and pd.notna(row['Year']):
        fire_zip_years.add((row['zip'], row['Year']))

base_table['had_fire'] = base_table.apply(
    lambda r: 1 if (r['zip'], r['year']) in fire_zip_years else 0, axis=1
)

print("Fire rate by year:")
for yr in sorted(base_table['year'].unique()):
    s = base_table[base_table['year'] == yr]
    print(f"  {int(yr)}: {s['had_fire'].sum():>4d} / {len(s):>5d} = {s['had_fire'].mean():.2%}")

Fire rate by year:
  2018:  224 /  2593 = 8.64%
  2019:  175 /  2593 = 6.75%
  2020:  250 /  2593 = 9.64%
  2021:  223 /  2593 = 8.60%
  2022:  173 /  2593 = 6.67%


In [5]:
# === Cell 5: Fire History Features ===

def compute_fire_history(row, fe):
    z, y = row['zip'], row['year']
    prior = fe[(fe['zip'] == z) & (fe['Year'] < y)]
    count = len(prior)
    total = prior['GIS_ACRES'].sum() if count > 0 else 0
    mx = prior['GIS_ACRES'].max() if count > 0 else 0
    return pd.Series({
        'prior_fire_count': count, 'prior_total_acres': total,
        'prior_max_acres': mx, 'had_prior_fire': 1 if count > 0 else 0,
        'prior_total_acres_log': np.log1p(total), 'prior_max_acres_log': np.log1p(mx),
    })

print("Computing fire history features...")
start = time.time()
fire_hist = base_table.apply(lambda r: compute_fire_history(r, fire_events), axis=1)
base_table = pd.concat([base_table, fire_hist], axis=1)
print(f"Complete ({time.time()-start:.1f}s)")

Computing fire history features...
Complete (2.9s)


In [6]:
# === Cell 6: Weather Features ===

weather_only = wildfire[wildfire['avg_tmax_c'].notna()].copy()
weather_only['year'] = weather_only['derived_year']
weather_only['month'] = weather_only['year_month'].str[5:7].astype(int)

def agg_weather(group):
    fs = group[group['month'].between(6, 10)]
    return pd.Series({
        'mean_tmax': group['avg_tmax_c'].mean(),
        'max_tmax': group['avg_tmax_c'].max(),
        'mean_tmin': group['avg_tmin_c'].mean(),
        'min_tmin': group['avg_tmin_c'].min(),
        'total_precip': group['tot_prcp_mm'].sum(),
        'mean_precip': group['tot_prcp_mm'].mean(),
        'min_precip': group['tot_prcp_mm'].min(),
        'temp_range': group['avg_tmax_c'].max() - group['avg_tmax_c'].min(),
        'fire_season_mean_tmax': fs['avg_tmax_c'].mean() if len(fs) > 0 else np.nan,
        'fire_season_max_tmax': fs['avg_tmax_c'].max() if len(fs) > 0 else np.nan,
        'fire_season_total_precip': fs['tot_prcp_mm'].sum() if len(fs) > 0 else np.nan,
    })

weather_agg = weather_only[weather_only['year'].isin([2018,2019,2020,2021])]\
    .groupby(['zip', 'year']).apply(agg_weather).reset_index()

base_table = base_table.merge(weather_agg, on=['zip', 'year'], how='left')

weather_2021 = weather_agg[weather_agg['year'] == 2021].drop(columns='year')
mask_2022 = base_table['year'] == 2022
weather_cols = [c for c in weather_agg.columns if c not in ['zip', 'year']]
for col in weather_cols:
    base_table.loc[mask_2022, col] = base_table.loc[mask_2022, 'zip'].map(
        weather_2021.set_index('zip')[col])

for col in weather_cols:
    base_table[col] = base_table[col].fillna(base_table.groupby('zip')[col].transform('mean'))
    base_table[col] = base_table[col].fillna(base_table[col].mean())

print(f"Weather features added. Remaining nulls: {base_table[weather_cols].isnull().sum().sum()}")

Weather features added. Remaining nulls: 0


In [7]:
# === Cell 7: Insurance and Census Features ===

sum_cols = [
    'Earned Premium', 'Earned Exposure',
    'CAT Cov A Fire -  Incurred Losses', 'CAT Cov A Fire -  Number of Claims',
    'Non-CAT Cov A Fire -  Incurred Losses', 'Non-CAT Cov A Fire -  Number of Claims',
    'Number of High Fire Risk Exposure', 'Number of Very High Fire Risk Exposure',
    'Number of Low Fire Risk Exposure', 'Number of Moderate Fire Risk Exposure',
    'Number of Negligible Fire Risk Exposure',
]
mean_cols = ['Avg Fire Risk Score', 'Cov A Amount Weighted Avg', 'Cov C Amount Weighted Avg']
census_cols = [
    'total_population', 'median_income', 'total_housing_units',
    'average_household_size', 'educational_attainment_bachelor_or_higher',
    'poverty_status', 'housing_value', 'housing_vacancy_number',
    'median_monthly_housing_costs', 'owner_occupied_housing_units',
    'renter_occupied_housing_units',
]

agg_dict = {}
for c in sum_cols:
    if c in insurance.columns: agg_dict[c] = 'sum'
for c in mean_cols:
    if c in insurance.columns: agg_dict[c] = 'mean'
for c in census_cols:
    if c in insurance.columns: agg_dict[c] = 'first'

ins_agg = insurance.groupby(['ZIP', 'Year']).agg(agg_dict).reset_index()
ins_agg = ins_agg.rename(columns={'ZIP': 'zip', 'Year': 'year'})
ins_agg['zip'] = ins_agg['zip'].astype(float)
ins_agg['year'] = ins_agg['year'].astype(float)

ppc_by_zip = insurance[insurance['Avg PPC'].notna()].groupby('ZIP')['Avg PPC'].mean()
ins_cols = list(agg_dict.keys())
base_table = base_table.merge(ins_agg, on=['zip', 'year'], how='left')
base_table['Avg PPC'] = base_table['zip'].map(ppc_by_zip)
base_table['Avg PPC'] = base_table['Avg PPC'].fillna(base_table['Avg PPC'].median())

ins_2021 = ins_agg[ins_agg['year'] == 2021].drop(columns='year')
for col in ins_cols:
    if col in ins_2021.columns and col in base_table.columns:
        base_table.loc[mask_2022, col] = base_table.loc[mask_2022, 'zip'].map(
            ins_2021.set_index('zip')[col])

claims_exp = [c for c in ins_cols if any(k in c for k in ['Claims','Losses','Exposure','Premium','Earned'])]
for c in claims_exp:
    if c in base_table.columns: base_table[c] = base_table[c].fillna(0)
for c in census_cols + mean_cols:
    if c in base_table.columns: base_table[c] = base_table[c].fillna(base_table[c].median())

for c in base_table.columns:
    if base_table[c].isnull().any() and base_table[c].dtype != 'object':
        base_table[c] = base_table[c].fillna(base_table[c].median())
        base_table[c] = base_table[c].fillna(0)

print(f"Feature matrix: {base_table.shape}")
print(f"Total nulls: {base_table.isnull().sum().sum()}")
print(f"\nFire rate by year:")
for yr in sorted(base_table['year'].unique()):
    s = base_table[base_table['year'] == yr]
    print(f"  {int(yr)}: {s['had_fire'].sum():>4d} / {len(s):>5d} = {s['had_fire'].mean():.2%}")

Feature matrix: (12965, 46)
Total nulls: 0

Fire rate by year:
  2018:  224 /  2593 = 8.64%
  2019:  175 /  2593 = 6.75%
  2020:  250 /  2593 = 9.64%
  2021:  223 /  2593 = 8.60%
  2022:  173 /  2593 = 6.67%


In [8]:
# === Cell 8: Prepare Training and Prediction Sets ===

exclude_cols = ['zip', 'year', 'had_fire']
feature_cols = [c for c in base_table.columns if c not in exclude_cols]

train_all = base_table[base_table['year'].isin([2018, 2019, 2020, 2021, 2022])]
X_train_all = train_all[feature_cols].values
y_train_all = train_all['had_fire'].values

scaler = StandardScaler()
X_train_all_s = scaler.fit_transform(X_train_all)

pred_2023_base = base_table[base_table['year'] == 2022].copy()
X_pred_2023 = pred_2023_base[feature_cols].values
X_pred_2023_s = scaler.transform(X_pred_2023)
pred_zips = pred_2023_base['zip'].values

print(f"Training: {X_train_all.shape[0]:,} rows (2018-2022), {X_train_all.shape[1]} features")
print(f"Fire rate: {y_train_all.mean():.2%}")
print(f"Prediction set: {X_pred_2023.shape[0]:,} zip codes")

Training: 12,965 rows (2018-2022), 43 features
Fire rate: 8.06%
Prediction set: 2,593 zip codes


In [9]:
# === Cell 9: Evaluation Helpers ===

def evaluate_clf(name, y_true, y_pred, y_prob=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    print(f"  {name}")
    print(f"    Accuracy={acc:.4f}  Precision={prec:.4f}  Recall={rec:.4f}  F1={f1:.4f}")
    print(f"    TN={cm[0][0]:>5d}  FP={cm[0][1]:>5d}  FN={cm[1][0]:>5d}  TP={cm[1][1]:>5d}")
    if y_prob is not None:
        print(f"    AUC-ROC={roc_auc_score(y_true, y_prob):.4f}")
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

def evaluate_reg(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape}

---
## Part 2: Classical Baselines (Task 1A)

In [10]:
# === Cell 10: Classical Models ===

scale_pos = (y_train_all == 0).sum() / (y_train_all == 1).sum()

xgb_final = XGBClassifier(n_estimators=200, max_depth=6, scale_pos_weight=scale_pos,
                           random_state=42, eval_metric='logloss')
xgb_final.fit(X_train_all_s, y_train_all)
xgb_2023_prob = xgb_final.predict_proba(X_pred_2023_s)[:, 1]
xgb_2023_pred = xgb_final.predict(X_pred_2023_s)

rf_final = RandomForestClassifier(n_estimators=200, max_depth=15, class_weight='balanced',
                                   random_state=42, n_jobs=-1)
rf_final.fit(X_train_all_s, y_train_all)
rf_2023_prob = rf_final.predict_proba(X_pred_2023_s)[:, 1]
rf_2023_pred = rf_final.predict(X_pred_2023_s)

print("Classical 2023 Predictions:")
print(f"  XGBoost:       {xgb_2023_pred.sum()} fires predicted ({xgb_2023_pred.mean():.2%})")
print(f"  Random Forest: {rf_2023_pred.sum()} fires predicted ({rf_2023_pred.mean():.2%})")

importance = pd.DataFrame({'feature': feature_cols, 'importance': xgb_final.feature_importances_})
importance = importance.sort_values('importance', ascending=False)
print(f"\nTop 10 Features (XGBoost):")
for _, row in importance.head(10).iterrows():
    print(f"  {row['feature']:<45s} {row['importance']:.4f}")

Classical 2023 Predictions:
  XGBoost:       195 fires predicted (7.52%)
  Random Forest: 300 fires predicted (11.57%)

Top 10 Features (XGBoost):
  Number of High Fire Risk Exposure             0.1687
  Avg PPC                                       0.0828
  CAT Cov A Fire -  Number of Claims            0.0761
  Earned Exposure                               0.0553
  prior_total_acres                             0.0290
  min_precip                                    0.0258
  Earned Premium                                0.0256
  Number of Very High Fire Risk Exposure        0.0248
  fire_season_mean_tmax                         0.0236
  mean_precip                                   0.0221


In [11]:
# === Cell 11: Validate Against Known 2023 Fire Zips ===

fire_2023 = fire_events[fire_events['Year'] == 2023]['zip'].unique()
fire_2023_set = set(fire_2023)
pred_zip_set = set(pred_zips)
overlap = fire_2023_set.intersection(pred_zip_set)

print(f"Known 2023 fire zips: {len(fire_2023)}")
print(f"Overlap with prediction set: {len(overlap)}")

for model_name, probs in [('XGBoost', xgb_2023_prob), ('Random Forest', rf_2023_prob)]:
    fire_probs = [probs[np.where(pred_zips == z)[0][0]] for z in overlap if z in pred_zip_set]
    nonfire_probs = [probs[np.where(pred_zips == z)[0][0]]
                     for z in pred_zips if z not in fire_2023_set][:500]
    print(f"\n{model_name}:")
    print(f"  Mean prob (fire zips):     {np.mean(fire_probs):.4f}")
    print(f"  Mean prob (non-fire zips): {np.mean(nonfire_probs):.4f}")
    print(f"  Separation ratio:          {np.mean(fire_probs)/np.mean(nonfire_probs):.1f}x")
    sorted_idx = np.argsort(probs)[::-1]
    for k in [100, 250, 500]:
        top_k = set(pred_zips[sorted_idx[:k]])
        captured = len(overlap.intersection(top_k))
        print(f"  Top-{k} capture: {captured}/{len(overlap)} ({captured/len(overlap):.0%})")

Known 2023 fire zips: 120
Overlap with prediction set: 118

XGBoost:
  Mean prob (fire zips):     0.5425
  Mean prob (non-fire zips): 0.0181
  Separation ratio:          30.0x
  Top-100 capture: 40/118 (34%)
  Top-250 capture: 74/118 (63%)
  Top-500 capture: 90/118 (76%)

Random Forest:
  Mean prob (fire zips):     0.5989
  Mean prob (non-fire zips): 0.0302
  Separation ratio:          19.8x
  Top-100 capture: 40/118 (34%)
  Top-250 capture: 74/118 (63%)
  Top-500 capture: 97/118 (82%)


---
## Part 3: Quantum Models (Task 1A)

Five quantum circuit variants trained on balanced subsamples:

| Model | Feature Map | Ansatz | Entanglement | Feature Order |
|---|---|---|---|---|
| Q1 | ZZFeatureMap | RealAmplitudes | Linear | Original (importance rank) |
| Q2 | ZZFeatureMap | RealAmplitudes | Linear | Domain-clustered |
| Q3 | WildfireQCircuit | Domain-informed | Literature-derived (14 CZ/layer) | Domain-clustered |
| Q4 | WildfireQCircuit | Domain-informed (2L) | Literature-derived (14 CZ/layer) | Domain-clustered |
| Q5 | Random | Random | Random (14 CZ/layer) | Domain-clustered |

In [12]:
# === Cell 12: Prepare Quantum Data ===

top10_original = [
    'Number of High Fire Risk Exposure', 'prior_max_acres', 'Avg PPC',
    'Earned Premium', 'min_precip', 'CAT Cov A Fire -  Number of Claims',
    'prior_total_acres', 'renter_occupied_housing_units',
    'educational_attainment_bachelor_or_higher', 'housing_vacancy_number',
]

REORDER = [6, 1, 0, 4, 3, 5, 2, 7, 8, 9]
top10_clustered = [top10_original[i] for i in REORDER]
CLUSTER_LABELS = ['Fire History','Fire History','Fire History','Weather',
                  'Insurance','Insurance','Insurance',
                  'Socioeconomic','Socioeconomic','Socioeconomic']

assert all(c in feature_cols for c in top10_original)

X_train_q_orig_s = StandardScaler().fit_transform(train_all[top10_original].values)
X_pred_q_orig_s = StandardScaler().fit(train_all[top10_original].values).transform(pred_2023_base[top10_original].values)

scaler_q_clust = StandardScaler()
X_train_q_clust_s = scaler_q_clust.fit_transform(train_all[top10_clustered].values)
X_pred_q_clust_s = scaler_q_clust.transform(pred_2023_base[top10_clustered].values)

rus = RandomUnderSampler(random_state=42)
X_bal_orig, y_bal_orig = rus.fit_resample(X_train_q_orig_s, y_train_all)
X_bal_clust, y_bal_clust = rus.fit_resample(X_train_q_clust_s, y_train_all)

print(f"Balanced training samples: {X_bal_orig.shape[0]}")
print(f"\nDomain-clustered qubit layout:")
for i, (name, cluster) in enumerate(zip(top10_clustered, CLUSTER_LABELS)):
    print(f"  q{i}: [{cluster:<15s}] {name}")

Balanced training samples: 2090

Domain-clustered qubit layout:
  q0: [Fire History   ] prior_total_acres
  q1: [Fire History   ] prior_max_acres
  q2: [Fire History   ] Number of High Fire Risk Exposure
  q3: [Weather        ] min_precip
  q4: [Insurance      ] Earned Premium
  q5: [Insurance      ] CAT Cov A Fire -  Number of Claims
  q6: [Insurance      ] Avg PPC
  q7: [Socioeconomic  ] renter_occupied_housing_units
  q8: [Socioeconomic  ] educational_attainment_bachelor_or_higher
  q9: [Socioeconomic  ] housing_vacancy_number


In [13]:
# === Cell 13: Circuit Builders ===

N_QUBITS = 10

def build_wildfire_feature_map(n_qubits=10, reps=2):
    """Domain-informed feature map derived from wildfire literature.
    Entanglement reflects known physical and socioeconomic interactions."""
    x = ParameterVector('x', n_qubits)
    qc = QuantumCircuit(n_qubits)
    for layer in range(reps):
        if layer == 0:
            for i in range(n_qubits): qc.h(i)
        for i in range(n_qubits): qc.ry(x[i], i)
        qc.cz(0,1); qc.cz(1,2); qc.cz(0,2)  # Fire History
        qc.cz(4,5); qc.cz(5,6); qc.cz(4,6)  # Insurance
        qc.cz(7,8); qc.cz(8,9); qc.cz(7,9)  # Socioeconomic
        qc.cz(3,0); qc.cz(3,2)               # Moisture-Fire History
        qc.cz(4,2)                            # Premium-Exposure
        qc.cz(8,2); qc.cz(7,2)               # Demographics-Exposure
    return qc

def build_wildfire_ansatz(n_qubits=10, n_layers=3):
    """Trainable ansatz mirroring the domain-informed entanglement topology."""
    n_params = n_qubits * (n_layers + 1)
    theta = ParameterVector('\u03b8', n_params)
    qc = QuantumCircuit(n_qubits)
    for layer in range(n_layers):
        off = layer * n_qubits
        for i in range(n_qubits): qc.ry(theta[off + i], i)
        qc.cz(0,1); qc.cz(1,2); qc.cz(0,2)
        qc.cz(4,5); qc.cz(5,6); qc.cz(4,6)
        qc.cz(7,8); qc.cz(8,9); qc.cz(7,9)
        qc.cz(3,0); qc.cz(3,2); qc.cz(4,2); qc.cz(8,2); qc.cz(7,2)
    off = n_layers * n_qubits
    for i in range(n_qubits): qc.ry(theta[off + i], i)
    return qc

def build_random_circuit(n_qubits=10, n_cz=14, reps=2, seed=123):
    """Random entanglement with identical gate count (ablation control)."""
    rng = np.random.RandomState(seed)
    all_pairs = [(i,j) for i in range(n_qubits) for j in range(i+1, n_qubits)]
    chosen_idx = rng.choice(len(all_pairs), n_cz, replace=False)
    pairs = [all_pairs[idx] for idx in chosen_idx]
    x = ParameterVector('x', n_qubits)
    qc = QuantumCircuit(n_qubits)
    for layer in range(reps):
        if layer == 0:
            for i in range(n_qubits): qc.h(i)
        for i in range(n_qubits): qc.ry(x[i], i)
        for (a,b) in pairs: qc.cz(a,b)
    return qc, pairs

def build_random_ansatz(n_qubits=10, n_layers=3, pairs=None):
    n_params = n_qubits * (n_layers + 1)
    theta = ParameterVector('\u03b8', n_params)
    qc = QuantumCircuit(n_qubits)
    for layer in range(n_layers):
        off = layer * n_qubits
        for i in range(n_qubits): qc.ry(theta[off + i], i)
        for (a,b) in pairs: qc.cz(a,b)
    off = n_layers * n_qubits
    for i in range(n_qubits): qc.ry(theta[off + i], i)
    return qc

print("Circuit builders defined.")

Circuit builders defined.


In [14]:
# === Cell 14: Train All 5 Quantum Models ===

quantum_results = {}
quantum_preds_2023 = {}
quantum_vqcs = {}

fm_rand, rand_pairs = build_random_circuit(10, n_cz=14, reps=2, seed=123)
an_rand = build_random_ansatz(10, n_layers=3, pairs=rand_pairs)

configs = [
    {'name': 'Q1: Baseline VQC (original order)',
     'fm': ZZFeatureMap(feature_dimension=10, reps=2, entanglement='linear'),
     'an': RealAmplitudes(num_qubits=10, reps=2, entanglement='linear'),
     'X_train': X_bal_orig, 'y_train': y_bal_orig,
     'X_pred': X_pred_q_orig_s, 'maxiter': 300},
    {'name': 'Q2: Reordered VQC (domain-clustered)',
     'fm': ZZFeatureMap(feature_dimension=10, reps=2, entanglement='linear'),
     'an': RealAmplitudes(num_qubits=10, reps=2, entanglement='linear'),
     'X_train': X_bal_clust, 'y_train': y_bal_clust,
     'X_pred': X_pred_q_clust_s, 'maxiter': 300},
    {'name': 'Q3: WildfireQCircuit (3 ansatz layers)',
     'fm': build_wildfire_feature_map(10, reps=2),
     'an': build_wildfire_ansatz(10, n_layers=3),
     'X_train': X_bal_clust, 'y_train': y_bal_clust,
     'X_pred': X_pred_q_clust_s, 'maxiter': 300},
    {'name': 'Q4: WildfireQCircuit Tuned (2 layers, 500 iter)',
     'fm': build_wildfire_feature_map(10, reps=2),
     'an': build_wildfire_ansatz(10, n_layers=2),
     'X_train': X_bal_clust, 'y_train': y_bal_clust,
     'X_pred': X_pred_q_clust_s, 'maxiter': 500},
    {'name': 'Q5: Random Entanglement (ablation)',
     'fm': fm_rand, 'an': an_rand,
     'X_train': X_bal_clust, 'y_train': y_bal_clust,
     'X_pred': X_pred_q_clust_s, 'maxiter': 300},
]

for i, cfg in enumerate(configs):
    print(f"\n{'='*65}")
    print(f"  [{i+1}/5] {cfg['name']}")
    print(f"  Parameters: {cfg['an'].num_parameters}, Iterations: {cfg['maxiter']}")
    print(f"  Training samples: {cfg['X_train'].shape[0]}")
    print(f"{'='*65}")
    print(f"  Started: {time.strftime('%Y-%m-%d %H:%M:%S')}")

    opt = COBYLA(maxiter=cfg['maxiter'])
    vqc = VQC(feature_map=cfg['fm'], ansatz=cfg['an'], optimizer=opt, num_qubits=10)

    t0 = time.time()
    vqc.fit(cfg['X_train'], cfg['y_train'])
    elapsed = time.time() - t0

    print(f"  Completed: {time.strftime('%Y-%m-%d %H:%M:%S')} ({elapsed/60:.1f} min)")

    pred_2023 = vqc.predict(cfg['X_pred'])
    quantum_preds_2023[cfg['name']] = pred_2023
    quantum_vqcs[cfg['name']] = vqc
    quantum_results[cfg['name']] = {
        'predictions': pred_2023.sum(), 'rate': pred_2023.mean(),
        'time_min': elapsed / 60, 'params': cfg['an'].num_parameters,
        'maxiter': cfg['maxiter'],
    }
    print(f"  2023 predictions: {pred_2023.sum()} fires ({pred_2023.mean():.2%})")

print(f"\n{'='*65}")
print(f"  All quantum models complete.")
print(f"{'='*65}")

No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.



  [1/5] Q1: Baseline VQC (original order)
  Parameters: 30, Iterations: 300
  Training samples: 2090
  Started: 2026-04-05 17:26:28
  Completed: 2026-04-05 21:48:39 (262.2 min)


No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.


  2023 predictions: 479 fires (18.47%)

  [2/5] Q2: Reordered VQC (domain-clustered)
  Parameters: 30, Iterations: 300
  Training samples: 2090
  Started: 2026-04-05 21:49:39
  Completed: 2026-04-06 02:31:21 (281.7 min)


No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.


  2023 predictions: 576 fires (22.21%)

  [3/5] Q3: WildfireQCircuit (3 ansatz layers)
  Parameters: 40, Iterations: 300
  Training samples: 2090
  Started: 2026-04-06 02:32:34
  Completed: 2026-04-06 07:55:46 (323.2 min)


No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.


  2023 predictions: 715 fires (27.57%)

  [4/5] Q4: WildfireQCircuit Tuned (2 layers, 500 iter)
  Parameters: 30, Iterations: 500
  Training samples: 2090
  Started: 2026-04-06 07:56:58
  Completed: 2026-04-06 14:17:14 (380.3 min)


No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.


  2023 predictions: 692 fires (26.69%)

  [5/5] Q5: Random Entanglement (ablation)
  Parameters: 40, Iterations: 300
  Training samples: 2090
  Started: 2026-04-06 14:18:10
  Completed: 2026-04-06 18:08:37 (230.4 min)
  2023 predictions: 784 fires (30.24%)

  All quantum models complete.


In [15]:
# === Cell 15: Quantum 2023 Validation ===

print("Quantum Model Validation Against Known 2023 Fire Zips")
print(f"Known fire zips in prediction set: {len(overlap)}\n")

for name, preds in quantum_preds_2023.items():
    captures = sum(preds[np.where(pred_zips == z)[0][0]] for z in overlap if z in pred_zip_set)
    print(f"  {name}")
    print(f"    Predicted fires: {preds.sum()}, Known fires captured: {captures}/{len(overlap)}")

Quantum Model Validation Against Known 2023 Fire Zips
Known fire zips in prediction set: 118

  Q1: Baseline VQC (original order)
    Predicted fires: 479, Known fires captured: 44/118
  Q2: Reordered VQC (domain-clustered)
    Predicted fires: 576, Known fires captured: 74/118
  Q3: WildfireQCircuit (3 ansatz layers)
    Predicted fires: 715, Known fires captured: 69/118
  Q4: WildfireQCircuit Tuned (2 layers, 500 iter)
    Predicted fires: 692, Known fires captured: 66/118
  Q5: Random Entanglement (ablation)
    Predicted fires: 784, Known fires captured: 73/118


In [16]:
# === Cell 16: Save 2023 Predictions ===

pred_df = pd.DataFrame({
    'ZIP': pred_zips.astype(int),
    'XGBoost_Probability': xgb_2023_prob,
    'XGBoost_Prediction': xgb_2023_pred,
    'RF_Probability': rf_2023_prob,
    'RF_Prediction': rf_2023_pred,
})
for name, preds in quantum_preds_2023.items():
    col = name.split(':')[0].strip()
    pred_df[f'{col}_Prediction'] = preds

pred_df['Risk_Tier'] = pd.cut(pred_df['XGBoost_Probability'],
    bins=[0, 0.1, 0.3, 0.5, 0.7, 1.0],
    labels=['Very Low', 'Low', 'Moderate', 'High', 'Very High'])
pred_df['Known_2023_Fire'] = pred_df['ZIP'].astype(float).isin(fire_2023_set).astype(int)
pred_df = pred_df.sort_values('XGBoost_Probability', ascending=False)
pred_df.to_csv('../data/task1_predictions_2023.csv', index=False)

print(f"Saved: {len(pred_df)} zip codes")
print(f"\nRisk Tier Distribution:")
print(pred_df['Risk_Tier'].value_counts().sort_index())
print(f"\nTop 20 Highest Risk Zip Codes:")
show_cols = [c for c in ['ZIP','XGBoost_Probability','Q1_Prediction','Q2_Prediction',
             'Q3_Prediction','Risk_Tier','Known_2023_Fire'] if c in pred_df.columns]
print(pred_df.head(20)[show_cols].to_string(index=False))

Saved: 2593 zip codes

Risk Tier Distribution:
Risk_Tier
Very Low     2278
Low            73
Moderate       47
High           14
Very High     181
Name: count, dtype: int64

Top 20 Highest Risk Zip Codes:
  ZIP  XGBoost_Probability  Q1_Prediction  Q2_Prediction  Q3_Prediction Risk_Tier  Known_2023_Fire
93619             0.999538              1              1              0 Very High                0
96001             0.999483              1              1              1 Very High                0
93675             0.999145              0              1              1 Very High                1
93446             0.998319              0              0              1 Very High                1
94565             0.998314              0              1              0 Very High                0
93306             0.998305              0              1              1 Very High                1
93420             0.998219              1              1              1 Very High                1
960

---
## Part 4: Task 2 — Insurance Premium Prediction

In [17]:
# === Cell 17: Quantum Fire Risk for Task 2 ===

# Use Q2 (best quantum model) to predict fire risk for 2020 zip codes
data_2020_raw = base_table[base_table['year'] == 2020][top10_clustered].values
zips_2020 = base_table[base_table['year'] == 2020]['zip'].values
data_2020_s = scaler_q_clust.transform(data_2020_raw)

# Use the Q2 VQC that was already trained
q2_vqc = quantum_vqcs['Q2: Reordered VQC (domain-clustered)']
q_pred_2020 = q2_vqc.predict(data_2020_s)

print(f"Quantum fire risk predictions for 2020: {q_pred_2020.sum()} fires ({len(q_pred_2020)} zips)")

Quantum fire risk predictions for 2020: 556 fires (2593 zips)


In [18]:
# === Cell 18: Task 2 Feature Matrix ===

ins_raw = pd.read_csv('../data/insurance_fire_census_weather_raw.csv')
agg_t2 = {}
for c in sum_cols:
    if c in ins_raw.columns: agg_t2[c] = 'sum'
for c in mean_cols:
    if c in ins_raw.columns: agg_t2[c] = 'mean'
for c in census_cols:
    if c in ins_raw.columns: agg_t2[c] = 'first'

zy = ins_raw.groupby(['ZIP', 'Year']).agg(agg_t2).reset_index()
ppc = ins_raw[ins_raw['Avg PPC'].notna()].groupby('ZIP')['Avg PPC'].mean()
zy['Avg PPC'] = zy['ZIP'].map(ppc).fillna(ppc.median())
zy = zy.sort_values(['ZIP', 'Year']).reset_index(drop=True)

lag_cols = ['Earned Premium', 'Earned Exposure',
            'CAT Cov A Fire -  Incurred Losses', 'CAT Cov A Fire -  Number of Claims',
            'Non-CAT Cov A Fire -  Incurred Losses', 'Non-CAT Cov A Fire -  Number of Claims']
for col in lag_cols:
    zy[f'{col}_lag1'] = zy.groupby('ZIP')[col].shift(1)
    zy[f'{col}_lag2'] = zy.groupby('ZIP')[col].shift(2)
zy['premium_growth'] = zy.groupby('ZIP')['Earned Premium'].pct_change()
zy['premium_rolling_avg'] = zy.groupby('ZIP')['Earned Premium'].transform(
    lambda x: x.shift(1).rolling(2, min_periods=1).mean())

fire_features = pd.read_csv('../data/feature_matrix_clean.csv')
fire_sub = fire_features[['zip','Year','had_fire','prior_fire_count','prior_total_acres_log',
                          'mean_tmax','fire_season_mean_tmax','fire_season_total_precip','total_precip']].copy()
fire_sub = fire_sub.rename(columns={'zip': 'ZIP'})
fire_sub['ZIP'] = fire_sub['ZIP'].astype(int)
zy = zy.merge(fire_sub, on=['ZIP', 'Year'], how='left')

q_2020_df = pd.DataFrame({'ZIP': zips_2020.astype(int), 'quantum_fire_risk': q_pred_2020})
q_2020_df['Year'] = 2021
zy = zy.merge(q_2020_df, on=['ZIP', 'Year'], how='left')
zy['quantum_fire_risk'] = zy['quantum_fire_risk'].fillna(0)
zy = zy.replace([np.inf, -np.inf], np.nan)

exclude_t2 = ['ZIP', 'Year', 'Earned Premium']
feat_cols_t2 = [c for c in zy.columns if c not in exclude_t2]
train_t2 = zy[zy['Year'] == 2020].dropna(subset=['Earned Premium'])
test_t2 = zy[zy['Year'] == 2021].dropna(subset=['Earned Premium'])
X_tr_t2 = train_t2[feat_cols_t2].fillna(0).replace([np.inf, -np.inf], 0)
y_tr_t2 = train_t2['Earned Premium']
X_te_t2 = test_t2[feat_cols_t2].fillna(0).replace([np.inf, -np.inf], 0)
y_te_t2 = test_t2['Earned Premium']
for c in feat_cols_t2:
    med = X_tr_t2[c].median()
    X_tr_t2[c] = X_tr_t2[c].fillna(med)
    X_te_t2[c] = X_te_t2[c].fillna(med)

print(f"Task 2: Train={X_tr_t2.shape}, Test={X_te_t2.shape}")
print(f"Quantum fire risk feature included: {'quantum_fire_risk' in feat_cols_t2}")

Task 2: Train=(2118, 47), Test=(2118, 47)
Quantum fire risk feature included: True


In [19]:
# === Cell 19: Task 2 Models ===

print("=" * 85)
print("  TASK 2: Insurance Premium Prediction (2021)")
print("=" * 85)
t2_results = {}

if 'Earned Premium_lag1' in X_te_t2.columns:
    r = evaluate_reg("Naive", y_te_t2.values, X_te_t2['Earned Premium_lag1'].values)
    t2_results['Naive'] = r
    print(f"  Naive:          MAE=${r['mae']:>10,.0f}  RMSE=${r['rmse']:>10,.0f}  R²={r['r2']:.4f}  MAPE={r['mape']:.1f}%")

rf_t2 = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_split=5,
                               min_samples_leaf=2, random_state=42, n_jobs=-1)
rf_t2.fit(X_tr_t2, y_tr_t2)
r = evaluate_reg("RF", y_te_t2.values, rf_t2.predict(X_te_t2))
t2_results['Random Forest'] = r
print(f"  Random Forest:  MAE=${r['mae']:>10,.0f}  RMSE=${r['rmse']:>10,.0f}  R²={r['r2']:.4f}  MAPE={r['mape']:.1f}%")

xgb_t2 = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1,
                        subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='rmse')
xgb_t2.fit(X_tr_t2, y_tr_t2)
r = evaluate_reg("XGB", y_te_t2.values, xgb_t2.predict(X_te_t2))
t2_results['XGBoost'] = r
print(f"  XGBoost:        MAE=${r['mae']:>10,.0f}  RMSE=${r['rmse']:>10,.0f}  R²={r['r2']:.4f}  MAPE={r['mape']:.1f}%")

xgb_imp = pd.DataFrame({'feature': feat_cols_t2, 'importance': xgb_t2.feature_importances_})
xgb_imp = xgb_imp.sort_values('importance', ascending=False)
print(f"\n  Top 10 features:")
for _, row in xgb_imp.head(10).iterrows():
    print(f"    {row['feature']:<45s} {row['importance']:.4f}")

  TASK 2: Insurance Premium Prediction (2021)
  Naive:          MAE=$   739,875  RMSE=$ 2,845,409  R²=0.7353  MAPE=44.0%
  Random Forest:  MAE=$   414,644  RMSE=$ 2,056,784  R²=0.8617  MAPE=31.1%
  XGBoost:        MAE=$   378,712  RMSE=$ 1,911,062  R²=0.8806  MAPE=307.2%

  Top 10 features:
    premium_rolling_avg                           0.6059
    Earned Premium_lag1                           0.1199
    Earned Exposure                               0.0633
    premium_growth                                0.0580
    Earned Premium_lag2                           0.0563
    Non-CAT Cov A Fire -  Incurred Losses         0.0198
    Number of Very High Fire Risk Exposure        0.0098
    Number of Low Fire Risk Exposure              0.0097
    mean_tmax                                     0.0086
    CAT Cov A Fire -  Number of Claims_lag1       0.0068


In [20]:
# === Cell 20: Save Task 2 Predictions ===

best_t2_name = 'XGBoost' if t2_results['XGBoost']['r2'] >= t2_results['Random Forest']['r2'] else 'Random Forest'
best_t2 = xgb_t2 if best_t2_name == 'XGBoost' else rf_t2
preds_t2 = pd.DataFrame({
    'ZIP': test_t2['ZIP'].values, 'Actual_Premium': y_te_t2.values,
    'Predicted_Premium': best_t2.predict(X_te_t2),
})
preds_t2['Absolute_Error'] = np.abs(preds_t2['Actual_Premium'] - preds_t2['Predicted_Premium'])
preds_t2.to_csv('../data/task2_predictions_2021_final.csv', index=False)
print(f"Task 2 best model: {best_t2_name}")
print(f"Predictions saved: {len(preds_t2)} zip codes")

Task 2 best model: XGBoost
Predictions saved: 2118 zip codes


---
## Part 5: Submission Summary

In [21]:
# === Cell 21: Complete Summary ===

print("#" * 80)
print("#  DELOITTE QUANTUM SUSTAINABILITY CHALLENGE 2026 — RESULTS")
print("#" * 80)

print(f"\n{'='*80}")
print(f"  TASK 1A: Wildfire Risk Prediction (2023)")
print(f"{'='*80}")
print(f"Training: 2018-2022 ({X_train_all.shape[0]:,} rows, {len(feature_cols)} features)")
print(f"Prediction: {len(pred_zips):,} California zip codes")
print(f"\nClassical:")
print(f"  XGBoost:       {xgb_2023_pred.sum():>4d} fires ({xgb_2023_pred.mean():.1%})")
print(f"  Random Forest: {rf_2023_pred.sum():>4d} fires ({rf_2023_pred.mean():.1%})")
print(f"\nQuantum:")
for name, res in quantum_results.items():
    print(f"  {name}: {res['predictions']:>4d} fires, "
          f"{res['params']} params, {res['time_min']:.1f} min")

print(f"\n{'='*80}")
print(f"  TASK 1B: Classical vs Quantum (development metrics, 2018-2020 train / 2021 test)")
print(f"{'='*80}")
print(f"\n{'Model':<50s} {'Acc':>6s} {'Prec':>6s} {'Rec':>6s} {'F1':>6s}")
print(f"{'-'*80}")
rows = [
    ('XGBoost (43 features)',                    0.889, 0.412, 0.665, 0.509),
    ('Random Forest (43 features)',              0.860, 0.351, 0.728, 0.473),
    ('XGBoost (top-10 features)',                0.904, 0.372, 0.710, 0.488),
    ('Q1: VQC baseline (original order)',        0.793, 0.203, 0.478, 0.285),
    ('Q2: VQC reordered (domain-clustered)',     0.789, 0.231, 0.621, 0.337),
    ('Q3: WildfireQCircuit (3L, 300i)',          0.627, 0.151, 0.714, 0.249),
    ('Q4: WildfireQCircuit tuned (2L, 500i)',    0.751, 0.189, 0.571, 0.284),
    ('Q5: Random entanglement (ablation)',       0.675, 0.163, 0.670, 0.262),
]
for name, acc, prec, rec, f1 in rows:
    print(f"{name:<50s} {acc:>6.3f} {prec:>6.3f} {rec:>6.3f} {f1:>6.3f}")

print(f"\n{'='*80}")
print(f"  TASK 2: Insurance Premium Prediction (2021)")
print(f"{'='*80}")
print(f"\n{'Model':<30s} {'MAE':>12s} {'RMSE':>12s} {'R²':>8s} {'MAPE':>8s}")
print(f"{'-'*75}")
for name, r in t2_results.items():
    print(f"{name:<30s} ${r['mae']:>10,.0f} ${r['rmse']:>10,.0f} {r['r2']:>7.4f} {r['mape']:>6.1f}%")

print(f"\n{'='*80}")
print(f"  QUANTUM RESOURCE REQUIREMENTS")
print(f"{'='*80}")
print(f"Platform:     IBM Qiskit {__import__('qiskit').__version__}")
print(f"Simulator:    Qiskit Aer (statevector)")
print(f"Qubits:       10")
print(f"Parameters:   30-40 trainable")
print(f"Training:     {X_bal_clust.shape[0]} balanced samples, COBYLA optimizer")
print(f"\nGitHub: https://github.com/Akhil-Kambhatla/deloitte-qsc-2026")

################################################################################
#  DELOITTE QUANTUM SUSTAINABILITY CHALLENGE 2026 — RESULTS
################################################################################

  TASK 1A: Wildfire Risk Prediction (2023)
Training: 2018-2022 (12,965 rows, 43 features)
Prediction: 2,593 California zip codes

Classical:
  XGBoost:        195 fires (7.5%)
  Random Forest:  300 fires (11.6%)

Quantum:
  Q1: Baseline VQC (original order):  479 fires, 30 params, 262.2 min
  Q2: Reordered VQC (domain-clustered):  576 fires, 30 params, 281.7 min
  Q3: WildfireQCircuit (3 ansatz layers):  715 fires, 40 params, 323.2 min
  Q4: WildfireQCircuit Tuned (2 layers, 500 iter):  692 fires, 30 params, 380.3 min
  Q5: Random Entanglement (ablation):  784 fires, 40 params, 230.4 min

  TASK 1B: Classical vs Quantum (development metrics, 2018-2020 train / 2021 test)

Model                                                 Acc   Prec    Rec     F1
------------------